<a href="https://colab.research.google.com/github/SandeepKisku24/lipur/blob/main/Lipur_Music_upload_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Music extractor for Lipur

In [1]:
!pip install -q yt-dlp langgraph langchain-google-genai pandas requests pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 3.8 MB/s eta 0:00:00


It's recommended to install a JavaScript runtime like `deno` for `yt-dlp` to function optimally and avoid potential missing formats. Run the following cell to install `deno`.

In [35]:
!curl -fsSL https://deno.land/x/install/install.sh | sh
# Add deno to PATH for current session
import os
os.environ["PATH"] += os.pathsep + os.path.expanduser("~/.deno/bin")
print("Deno installed and added to PATH.")

######################################################################## 100.0%
Archive:  /root/.deno/bin/deno.zip
  inflating: /root/.deno/bin/deno    
Installed dx alias, if this conflicts with an existing command, you can remove it with `rm $(which dx)` and choose a new name with `dx --install-alias <new-name>`
Deno was installed successfully to /root/.deno/bin/deno
sh: 109: cannot open /dev/tty: No such device or address
Deno installed and added to PATH.


In [5]:
import pandas as pd

In [29]:
songs = pd.read_excel("/content/Lipur List.xlsx")
songs.columns = songs.columns.str.strip() # Clean up column names
print(songs)

                                           url
0  https://www.youtube.com/watch?v=SWIUff-1FYM
1  https://www.youtube.com/watch?v=VFnDYiJSDxY


In [30]:
song_list = pd.DataFrame(songs)
print(song_list.iloc[0])

url    https://www.youtube.com/watch?v=SWIUff-1FYM
Name: 0, dtype: object


In [31]:
first_url = song_list.iloc[0]['url']
print(f"Processing URL: {first_url}")

Processing URL: https://www.youtube.com/watch?v=SWIUff-1FYM


In [41]:
import yt_dlp

def get_video_info(url):
    ydl_opts = {
        'quiet': True,
        'simulate': True, # Only simulate, do not download
        'format': 'bestaudio/best', # Get best audio format info
        'extract_flat': True, # Do not extract playlists
        'force_generic_extractor': False,
        'cookiefile': '/content/cookies.txt', # Added to handle YouTube login/bot detection
        'remote_components': ['ejs:github'] # Add this to enable remote JS challenge solvers
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False) # download=False to just get info
        return info

video_info = get_video_info(first_url)
if video_info:
    print(f"Title: {video_info.get('title')}")
    print(f"Uploader: {video_info.get('uploader')}")
    print(f"Thumbnail URL: {video_info.get('thumbnail')}")
else:
    print(f"Could not retrieve information for {first_url}")

Title: ESEL KURI || NEW HIT SANTHALI VIDEO SONG || TOM MURMU || 2018©
Uploader: Tom Murmu
Thumbnail URL: https://i.ytimg.com/vi/SWIUff-1FYM/maxresdefault.jpg


This code snippet uses `yt-dlp` to extract metadata from the YouTube URL without actually downloading the video. We can use this information to populate fields like `title`, `coverUrl`, and `artists`. Next, we can integrate this into a loop to process all URLs and then focus on downloading the MP3s and generating genres.

In [42]:
import yt_dlp
import os

# Create a temporary folder if it doesn't exist
os.makedirs("./tmp_downloads", exist_ok=True)

def get_video_info_and_download(url):
    ydl_opts = {
        'quiet': True,
        'format': 'bestaudio/best',
        'extract_flat': False, # Changed to False to ensure we get full description & date
        'cookiefile': '/content/cookies.txt',
        'remote_components': ['ejs:github'],

        # --- MISSING PIECES ADDED HERE ---
        'outtmpl': './tmp_downloads/temp_track', # Where to save the file
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }]
    }

    # Changed download=False to download=True
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        return info

# Test it
first_url = "https://www.youtube.com/watch?v=SWIUff-1FYM" # Replace with your test link
video_info = get_video_info_and_download(first_url)

if video_info:
    print("--- METADATA EXTRACTED ---")
    print(f"Title: {video_info.get('title')}")
    print(f"Thumbnail URL: {video_info.get('thumbnail')}")

    # Extract the Year (yt-dlp returns YYYYMMDD, so we slice the first 4 characters)
    raw_date = video_info.get('upload_date')
    print(f"Created Year: {raw_date[:4] if raw_date else '2026'}")

    # Get the description for the AI
    desc = video_info.get('description')
    print(f"Description (First 100 chars): {desc[:100] if desc else 'None'}...")

    print("\n--- FILE STATUS ---")
    print("File downloaded to: ./tmp_downloads/temp_track.mp3")
else:
    print(f"Could not retrieve information for {first_url}")

--- METADATA EXTRACTED ---
Title: ESEL KURI || NEW HIT SANTHALI VIDEO SONG || TOM MURMU || 2018©
Thumbnail URL: https://i.ytimg.com/vi/SWIUff-1FYM/maxresdefault.jpg
Created Year: 2018
Description (First 100 chars): Here we come up with yet another Music Video of our album "ESEL KURI". 
We have a special competitio...

--- FILE STATUS ---
File downloaded to: ./tmp_downloads/temp_track.mp3


### Step 1: Define Song Metadata Structure with Pydantic

To ensure consistency and define the mandatory fields, let's create a Pydantic model for our song metadata. This will help us structure the data as it moves through our LangGraph workflow.

In [43]:
from pydantic import BaseModel, Field, HttpUrl
from typing import Optional, List

class SongMetadata(BaseModel):
    url: HttpUrl = Field(..., description="Original YouTube or SoundCloud URL of the song.")
    title: str = Field(..., description="Cleaned title of the song.")
    coverUrl: HttpUrl = Field(..., description="URL of the song's cover image.")
    genre: str = Field(..., description="Genre of the song (LLM-generated).")
    created_year: str = Field(..., description="Year the song was created or uploaded (default if not found).")
    uploadUser: str = Field("Agent", description="User responsible for uploading the song.")
    file_path: Optional[str] = Field(None, description="Local path to the downloaded MP3 file.")
    artists: List[str] = Field(default_factory=list, description="List of artists associated with the song.")
    song_id: Optional[str] = Field(None, description="ID of the song in the music application after upload.")
    request_id: Optional[str] = Field(None, description="Unique ID for the upload request.")


# Example usage:
# song_data = SongMetadata(url="https://www.youtube.com/watch?v=SWIUff-1FYM", title="ESEL KURI", coverUrl="https://example.com/cover.jpg", genre="Santhali", created_year="2018", artists=["Tom Murmu"])
# print(song_data.model_dump_json(indent=2))

### Step 2: Outline the LangGraph Workflow

Now that we have a structured way to handle our song metadata, let's outline the different 'nodes' or 'agents' we'll need in our LangGraph workflow to accomplish all the tasks you've described. This will help us break down the problem into manageable components.

Here's a proposed workflow:

1.  **`ExtractMetadataNode`**: Takes a song URL, uses `yt-dlp` (or similar for SoundCloud) to extract raw metadata like title, uploader, thumbnail, and upload date. It will output a `SongMetadata` object, albeit with some fields still raw or empty.
2.  **`CleanTitleNode`**: Takes the `SongMetadata` object, uses an LLM to clean and purify the `title`, removing extraneous phrases like 'new song', 'mp3', etc.
3.  **`GenerateGenreNode`**: Takes the `SongMetadata` object and possibly the description, uses an LLM to infer and assign a `genre`.
4.  **`CheckDuplicateNode`**: Calls your backend API (`https://lipur-backend.onrender.com/songs`) to check if a song with a similar title and artist already exists to prevent duplicates.
5.  **`DownloadMP3Node`**: If no duplicate is found, it downloads the MP3 file using `yt-dlp` and renames it using the cleaned title, updating the `file_path` in the `SongMetadata` object.
6.  **`UploadToAPINode`**: Uploads the downloaded MP3 and the `SongMetadata` to your backend API, obtaining a `song_id`.
7.  **`UpdateExcelNode`**: Writes the final status, `song_id`, `request_id`, and other relevant details back to your Excel sheet (`Lipur List.xlsx`).

We'll use LangGraph to orchestrate these nodes, allowing for conditional logic (e.g., skip download/upload if a duplicate is found). We'll also need to manage the state (the `SongMetadata` object) as it passes between these nodes.

### Step 3: Initialize LangGraph and Define Graph State

Let's set up the basic LangGraph structure and define the graph state, which will be our `SongMetadata` object.

In [44]:
from langgraph.graph import StateGraph, END

# The state of our graph will be the SongMetadata object
# We'll also include a 'status' field to track the processing outcome.
class GraphState(BaseModel):
    song_metadata: SongMetadata
    status: str = Field("pending", description="Current status of the song processing (pending, downloaded, uploaded, failed).")
    error_message: Optional[str] = Field(None, description="Error message if processing fails.")

# Initialize the StateGraph
workflow = StateGraph(GraphState)

print("LangGraph workflow initialized with GraphState.")

LangGraph workflow initialized with GraphState.


### Step 4: Implement `ExtractMetadataNode`

This node will leverage `yt-dlp` to get the initial metadata from the YouTube URL. It will populate the `SongMetadata` object with the raw title, cover URL, and a preliminary artist from the uploader. The `created_year` will also be extracted here. The `genre` and cleaned `title` will be handled by subsequent LLM-based nodes.

In [45]:
import yt_dlp
from typing import Dict, Any

def extract_metadata_node(state: GraphState) -> GraphState:
    print("--- Entering ExtractMetadataNode ---")
    song_metadata = state.song_metadata
    url = str(song_metadata.url)

    ydl_opts = {
        'quiet': True,
        'simulate': True, # Only simulate, do not download MP3 yet
        'format': 'bestaudio/best',
        'extract_flat': False, # Changed to False to ensure full description and date
        'cookiefile': '/content/cookies.txt',
        'remote_components': ['ejs:github'],
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=False) # Only extract info

        if info:
            # Populate SongMetadata object
            song_metadata.title = info.get('title', 'Unknown Title')
            song_metadata.coverUrl = HttpUrl(info.get('thumbnail', '')) if info.get('thumbnail') else HttpUrl("http://example.com/default_cover.jpg") # Provide a default if None

            # Extract uploader as initial artist
            uploader = info.get('uploader')
            if uploader:
                song_metadata.artists = [uploader]

            # Extract Created Year from upload_date (YYYYMMDD)
            upload_date = info.get('upload_date')
            if upload_date and len(upload_date) >= 4:
                song_metadata.created_year = upload_date[:4]
            else:
                song_metadata.created_year = 'Unknown'

            # Store description for potential LLM use later
            song_metadata.description = info.get('description', '')

            state.song_metadata = song_metadata
            state.status = "metadata_extracted"
            print(f"Metadata extracted for: {song_metadata.title}")
        else:
            state.status = "failed"
            state.error_message = f"Could not retrieve information for {url}"
            print(f"Failed to extract metadata for {url}")

    except yt_dlp.DownloadError as e:
        state.status = "failed"
        state.error_message = f"yt-dlp DownloadError: {e}"
        print(f"yt-dlp DownloadError: {e}")
    except Exception as e:
        state.status = "failed"
        state.error_message = f"An unexpected error occurred: {e}"
        print(f"An unexpected error occurred: {e}")

    return state


# Add the node to the workflow
workflow.add_node("extract_metadata", extract_metadata_node)
print("ExtractMetadataNode added to workflow.")

ExtractMetadataNode added to workflow.


### Step 5: Implement `ProcessMetadataNode` with LLM

This node will use the Gemini LLM to:
1.  **Clean the song title**: Remove any extraneous information like 'NEW HIT', 'MP3', years, or copyright notices.
2.  **Extract Artists**: Identify key artists from the title or description.
3.  **Infer Genre**: Determine a suitable genre for the song based on its title, uploader, and description.

The LLM will be instructed to return its output in a JSON format for structured updates to our `SongMetadata` object.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from google.colab import userdata

# Configure Gemini API
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(model="gemini-pro", google_api_key=GOOGLE_API_KEY)

# Define the prompt for the LLM
metadata_processing_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert music metadata processor. Your task is to clean song titles, identify artists, and infer genres from provided song information. Always output a JSON object with 'cleaned_title', 'extracted_artists' (list of strings), and 'inferred_genre' (single string)."),
    ("human", "Process the following song details:\n\nTitle: {raw_title}\nUploader: {uploader}\nDescription: {description}\n\nExtract the cleanest possible title, a list of primary artists, and an appropriate genre. Prioritize artist names from the 'uploader' if available and reasonable. If no specific genre is clear, provide a general one like 'World Music', 'Pop', 'Folk', etc.\n\nJSON Output:")
])

# Create the LLM chain
metadata_processing_chain = metadata_processing_prompt | llm | JsonOutputParser()

def process_metadata_node(state: GraphState) -> GraphState:
    print("--- Entering ProcessMetadataNode (LLM) ---")
    song_metadata = state.song_metadata

    try:
        # Prepare input for the LLM
        raw_title = song_metadata.title
        uploader = song_metadata.artists[0] if song_metadata.artists else "Unknown Artist"
        description = song_metadata.description

        llm_response = metadata_processing_chain.invoke({
            "raw_title": raw_title,
            "uploader": uploader,
            "description": description
        })

        # Update SongMetadata with LLM's output
        song_metadata.title = llm_response.get("cleaned_title", raw_title)
        # Ensure artists list from LLM is used, or default to existing uploader if LLM doesn't provide
        if llm_response.get("extracted_artists"):
            song_metadata.artists = llm_response["extracted_artists"]
        song_metadata.genre = llm_response.get("inferred_genre", "Unknown Genre")

        state.song_metadata = song_metadata
        state.status = "metadata_processed_llm"
        print(f"LLM processed metadata for: {song_metadata.title}")

    except Exception as e:
        state.status = "failed"
        state.error_message = f"LLM processing error: {e}"
        print(f"LLM processing error: {e}")

    return state

# Add the node to the workflow
workflow.add_node("process_metadata_llm", process_metadata_node)
print("ProcessMetadataNode (LLM) added to workflow.")